In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer,AutoConfig,AutoModel
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# decoding
其中由于贪心搜索过于重复性，因此使用beam

In [ ]:
from transformers import AutoModelForCausalLM

In [ ]:
model_ckpt = 'gpt2-xl'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForCausalLM.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
sample_text = 'A long long time ago, '

In [ ]:
model_inputs = tokenizer(sample_text,return_tensors='pt')
model_inputs

{'input_ids': tensor([[  32,  890,  890,  640, 2084,   11,  220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}

# greedy search


In [ ]:
input_ids = model_inputs['input_ids']
print(input_ids.shape)
input_ids

torch.Size([1, 7])


tensor([[  32,  890,  890,  640, 2084,   11,  220]])

In [ ]:
sorted_ids = torch.argsort(torch.softmax(model(input_ids).logits[0,-1,:],dim=-1),dim=-1,descending=True)
sorted_ids[None,0,None].shape

torch.Size([1, 1])

In [20]:
n_steps = 10
# 步长10，且选出top5
choices_per_step = 5

iterations = []
with torch.no_grad():
  for _ in range(n_steps):
    # 每次都选出5个候选
    iteration = {}
    iteration['input'] = tokenizer.decode(input_ids[0])

    output = model(input_ids = input_ids)

    last_token_logits = output.logits[0,-1,:]
    last_token_probs = torch.softmax(last_token_logits,dim=-1)
    sorted_ids = torch.argsort(last_token_probs,dim=-1,descending=True)

    for choice_idx in range(choices_per_step):
      token_id = sorted_ids[choice_idx]
      token_prob = last_token_probs[token_id].cpu().numpy()
      token_choice = f'{tokenizer.decode(token_id)}({100*token_prob:.2f}%)'
      iteration[f'choice {choice_idx+1}'] = token_choice

    print('before append input_ids.shape',input_ids.shape)
    input_ids = torch.cat([input_ids,sorted_ids[None,0,None]],dim=-1)
    print('after append input_ids.sahpe',input_ids.shape)

    iterations.append(iteration)

before append input_ids.shape torch.Size([1, 17])
after append input_ids.sahpe torch.Size([1, 18])
before append input_ids.shape torch.Size([1, 18])
after append input_ids.sahpe torch.Size([1, 19])
before append input_ids.shape torch.Size([1, 19])
after append input_ids.sahpe torch.Size([1, 20])
before append input_ids.shape torch.Size([1, 20])
after append input_ids.sahpe torch.Size([1, 21])
before append input_ids.shape torch.Size([1, 21])
after append input_ids.sahpe torch.Size([1, 22])
before append input_ids.shape torch.Size([1, 22])
after append input_ids.sahpe torch.Size([1, 23])
before append input_ids.shape torch.Size([1, 23])
after append input_ids.sahpe torch.Size([1, 24])
before append input_ids.shape torch.Size([1, 24])
after append input_ids.sahpe torch.Size([1, 25])
before append input_ids.shape torch.Size([1, 25])
after append input_ids.sahpe torch.Size([1, 26])
before append input_ids.shape torch.Size([1, 26])
after append input_ids.sahpe torch.Size([1, 27])


In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(iterations)

,input,choice 1,choice 2,choice 3,choice 4,choice 5
0,"A long long time ago,",(47.70%),『(8.94%),_____(3.93%),____(3.67%),________(3.51%)
1,"A long long time ago,",I(19.67%),in(6.90%),a(6.87%),the(5.73%),there(3.60%)
2,"A long long time ago, I",was(13.66%),had(8.50%),wrote(5.17%),read(3.13%),used(2.92%)
3,"A long long time ago, I was",a(13.13%),in(6.68%),on(2.30%),at(2.20%),asked(2.11%)
4,"A long long time ago, I was a",young(3.32%),little(2.78%),very(2.55%),kid(2.51%),child(2.14%)
5,"A long long time ago, I was a young",man(12.78%),girl(10.55%),boy(7.64%),",(4.53%)",lad(4.40%)
6,"A long long time ago, I was a young man",",(19.50%)",.(14.43%),who(8.64%),and(8.21%),with(6.17%)
7,"A long long time ago, I was a young man,",and(17.90%),a(4.82%),just(3.76%),living(3.62%),I(3.08%)
8,"A long long time ago, I was a young man, and",I(33.80%),a(7.22%),my(5.96%),in(3.00%),the(2.45%)
9,"A long long time ago, I was a young man, and I",was(25.14%),had(14.60%),wanted(2.78%),thought(2.11%),used(1.89%)


In [21]:
iterations[-1]

{'input': 'A long long time ago, \xa0I was a young man, and I was a student at the University of California, Berkeley',
 'choice 1': '.(71.09%)',
 'choice 2': ',(15.76%)',
 'choice 3': ' .(2.49%)',
 'choice 4': ' in(1.11%)',
 'choice 5': ' ((1.04%)'}

In [24]:
def greedy_search(model,input_ids,max_steps=10,max_choices=5):
  iteartions = []
  input_ids_clone = input_ids.clone()
  with torch.no_grad():
    for _ in range(max_steps):
      iteration = {}
      iteration['input'] = tokenizer.decode(input_ids_clone[0])

      output = model(input_ids=input_ids_clone)

      last_token_logits = output.logits[0,-1,:]
      last_token_probs = torch.softmax(last_token_logits,dim=-1)
      sorted_ids = torch.argsort(last_token_probs,dim=-1,descending=True)

    for choice_idx in range(choices_per_step):
      token_id = sorted_ids[choice_idx]
      token_prob = last_token_probs[token_id].cpu().numpy()
      token_choice = f'{tokenizer.decode(token_id)}({100*token_prob:.2f}%)'
      iteration[f'choice {choice_idx+1}'] = token_choice

    input_ids_clone = torch.cat([input_ids_clone,sorted_ids[None,0,None]],dim=-1)

    iterations.append(iteration)
  return iterations

In [25]:
pd.DataFrame(greedy_search(model,input_ids, ))

,input,choice 1,choice 2,choice 3,choice 4,choice 5
0,"A long long time ago, I was a young man, and ...",a(7.94%),in(4.89%),very(2.89%),looking(1.90%),not(1.76%)
1,"A long long time ago, I was a young man, and ...",student(3.16%),very(2.79%),young(2.76%),member(2.09%),little(1.88%)
2,"A long long time ago, I was a young man, and ...",at(30.31%),.(22.40%),of(21.10%),in(7.87%),",(4.85%)"
3,"A long long time ago, I was a young man, and ...",the(28.17%),a(18.46%),an(2.87%),Harvard(1.33%),(1.22%)
4,"A long long time ago, I was a young man, and ...",University(41.26%),university(3.14%),time(2.29%),local(2.20%),College(1.70%)
5,"A long long time ago, I was a young man, and ...",of(93.18%),.(2.63%),in(0.81%),",(0.49%)",Of(0.43%)
6,"A long long time ago, I was a young man, and ...",California(4.62%),Texas(3.75%),Illinois(3.74%),Chicago(3.57%),Michigan(3.13%)
7,"A long long time ago, I was a young man, and ...",",(38.74%)",at(22.38%),.(15.69%),in(6.19%),Berkeley(2.36%)
8,"A long long time ago, I was a young man, and ...",Berkeley(57.35%),Santa(10.84%),San(5.83%),Los(5.57%),Davis(4.91%)
9,"A long long time ago, I was a young man, and ...",.(71.09%),",(15.76%)",.(2.49%),in(1.11%),((1.04%)
